# Day 3 — Vector Databases and Advanced RAG Engineering

Full RAG pipeline: document chunking → ChromaDB (HNSW) → BM25 → RRF hybrid search → cross-encoder reranking → RAGAS-style evaluation.

In [16]:
!pip install chromadb sentence-transformers rank-bm25 numpy

In [17]:
import re
import numpy as np
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

In [18]:
# =====================================================================
# KNOWLEDGE BASE — Eight data-engineering documents

In [19]:
!wget -O Books.csv "https://gist.githubusercontent.com/jaidevd/23aef12e9bf56c618c41/raw/c05e98672b8d52fa0cb94aad80f75eb78342e5d4/books.csv"

--2026-07-22 16:43:12--  https://gist.githubusercontent.com/jaidevd/23aef12e9bf56c618c41/raw/c05e98672b8d52fa0cb94aad80f75eb78342e5d4/books.csv
Resolving gist.githubusercontent.com (gist.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to gist.githubusercontent.com (gist.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12151 (12K) [text/plain]
Saving to: ‘Books.csv’

Books.csv           100%[===================>]  11.87K  --.-KB/s    in 0s      

2026-07-22 16:43:13 (71.8 MB/s) - ‘Books.csv’ saved [12151/12151]



In [20]:
# =====================================================================

import pandas as pd
books=pd.read_csv("Books.csv")
DOCUMENTS=[]
for i,row in books.iterrows():
    DOCUMENTS.append({
        "id":f"book_{i}",
"text": f"Title: {row['Title']}. Author: {row['Author']}. Genre: {row['Genre']}. Height: {row['Height']}. Publisher: {row['Publisher']}"
    })


In [21]:
# =====================================================================
# STAGE 1 — CHUNKING (sentence-level with overlap)

In [22]:
# =====================================================================

def chunk_documents(docs: list[dict], chunk_size: int = 2) -> list[dict]:
    """
    Splits each document into overlapping sentence chunks.
    chunk_size = 2 means each chunk contains 2 consecutive sentences.
    Overlap of 1 sentence ensures context is not lost at chunk boundaries.
    """
    all_chunks = []
    for doc in docs:
        sentences = re.split(r"(?<=[.!?])\s+", doc["text"].strip())
        for i in range(0, len(sentences), max(1, chunk_size - 1)):
            chunk_text = " ".join(sentences[i : i + chunk_size])
            if not chunk_text.strip():
                continue
            all_chunks.append({
                "id":     f"{doc['id']}_chunk_{i:03d}",
                "text":   chunk_text,
                "doc_id": doc["id"],
            })
    return all_chunks

In [23]:
# =====================================================================
# STAGE 2 — VECTOR INDEX (ChromaDB + SentenceTransformer bi-encoder)

In [24]:
# =====================================================================

def build_vector_index(chunks: list[dict]) -> chromadb.Collection:
    """
    Embeds all chunks with all-MiniLM-L6-v2 and stores them in ChromaDB.
    ChromaDB uses HNSW internally — the same index type as Pinecone/Weaviate.
    """
    print("\n📦 Building ChromaDB vector index...")
    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client     = chromadb.Client()
    collection = client.get_or_create_collection(
        "rag_lab_knowledge_base", embedding_function=ef
    )
    collection.add(
        ids       = [c["id"]   for c in chunks],
        documents = [c["text"] for c in chunks],
        metadatas = [{"doc_id": c["doc_id"]} for c in chunks],
    )
    print(f"   ✅ {len(chunks)} chunks indexed (HNSW backend, all-MiniLM-L6-v2 embeddings)")
    return collection

In [25]:
# =====================================================================
# STAGE 3 — BM25 KEYWORD INDEX

In [26]:
# =====================================================================

class BM25Index:
    """
    Classic TF-IDF keyword search using the BM25 ranking function.
    Finds exact keyword matches that semantic search often misses
    (e.g., product names, acronyms, version numbers like 'GPT-4').
    """

    def __init__(self, chunks: list[dict]):
        tokenised  = [c["text"].lower().split() for c in chunks]
        self.bm25  = BM25Okapi(tokenised)
        self.chunks = chunks

    def search(self, query: str, top_k: int = 10) -> list[tuple[float, dict]]:
        scores = self.bm25.get_scores(query.lower().split())
        ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)
        return [(score, self.chunks[idx]) for idx, score in ranked[:top_k]]

In [27]:
# =====================================================================
# STAGE 4 — HYBRID SEARCH WITH RECIPROCAL RANK FUSION

In [28]:
# =====================================================================

def reciprocal_rank_fusion(
    vector_hits: list[dict],
    bm25_hits:   list[tuple[float, dict]],
    k:           int = 60,
    top_k:       int = 6,
) -> list[dict]:
    """
    RRF score = Σ 1/(k + rank_i) across both result lists.

    k=60 is the empirically optimal constant — it prevents the top-rank
    position from dominating and rewards documents that rank consistently
    across both systems. No weights to tune: RRF is parameter-free.
    """
    rrf_scores: dict[str, float] = {}
    id_to_chunk: dict[str, dict] = {}

    for rank, hit in enumerate(vector_hits):
        cid = hit["id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
        id_to_chunk[cid] = {"id": cid, "text": hit["document"]}

    for rank, (_, chunk) in enumerate(bm25_hits):
        cid = chunk["id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (k + rank + 1)
        id_to_chunk[cid] = chunk

    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)
    return [id_to_chunk[cid] for cid in sorted_ids[:top_k]]

In [29]:
# =====================================================================
# STAGE 5 — CROSS-ENCODER RERANKING (Stage 2 precision gate)

In [30]:
# =====================================================================

def rerank(
    query:       str,
    candidates:  list[dict],
    model_name:  str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
    top_k:       int = 3,
) -> list[dict]:
    """
    Cross-encoder jointly encodes the (query, document) pair.
    This is much more accurate than cosine similarity between
    independent embeddings, because it can model the interaction
    between query tokens and document tokens directly.

    Cost: O(n) inference calls vs O(1) for bi-encoder similarity.
    Strategy: retrieve top-50 fast with bi-encoder, rerank top-50
    with cross-encoder, return top-5 to the LLM.
    """
    print(f"  🎯 Cross-encoder reranking {len(candidates)} candidates...")
    model  = CrossEncoder(model_name)
    pairs  = [(query, c["text"]) for c in candidates]
    scores = model.predict(pairs)
    ranked = sorted(zip(scores, candidates), key=lambda x: x[0], reverse=True)
    return [doc for _, doc in ranked[:top_k]]

In [31]:
# =====================================================================
# STAGE 6 — RAG PROMPT CONSTRUCTION

In [32]:
# =====================================================================

def build_rag_prompt(query: str, context_docs: list[dict]) -> str:
    """
    Constructs the prompt sent to the LLM.
    In production: replace the print statement with an actual API call
    (OpenAI, Anthropic, or any OpenRouter-compatible model).
    """
    context = "\n\n".join(
        f"[Source {i+1}]: {d['text']}" for i, d in enumerate(context_docs)
    )
    return (
        "You are a data engineering expert. Answer the question strictly based\n"
        "on the provided context. Do not add information not in the context.\n\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {query}\n\n"
        "ANSWER:"
    )

In [33]:
# =====================================================================
# STAGE 7 — EVALUATION (simplified RAGAS-style metrics)

In [34]:
# =====================================================================

def evaluate(
    query:          str,
    retrieved_docs: list[dict],
    embed_model:    SentenceTransformer,
    relevance_threshold: float = 0.30,
) -> dict:
    """
    Computes two simplified metrics using cosine similarity:

    Context Precision = fraction of retrieved chunks above the
                        relevance threshold (no noise in context).
    Avg Similarity    = mean cosine score between query and chunks
                        (proxy for Context Recall).

    In production: use RAGAS which calls an LLM judge for all four metrics.
    Alert if faithfulness < 0.85 — it means the LLM is hallucinating.
    """
    q_emb   = embed_model.encode(query, normalize_embeddings=True)
    scores  = [
        float(np.dot(q_emb, embed_model.encode(d["text"], normalize_embeddings=True)))
        for d in retrieved_docs
    ]
    relevant = sum(s > relevance_threshold for s in scores)
    return {
        "context_precision": round(relevant / len(scores), 3),
        "avg_similarity":    round(sum(scores) / len(scores), 3),
        "chunks_in_context": len(retrieved_docs),
    }

In [35]:
# =====================================================================
# MAIN — Full pipeline run

In [36]:
# =====================================================================

def main():
    print("=" * 65)
    print("  Day 3 Lab — Vector Databases and Advanced RAG Engineering")
    print("=" * 65)

    # Stage 1: Chunk
    chunks = chunk_documents(DOCUMENTS, chunk_size=2)
    print(f"\n📄 {len(DOCUMENTS)} documents → {len(chunks)} chunks after splitting")

    # Stage 2: Build indexes
    collection  = build_vector_index(chunks)
    bm25_index  = BM25Index(chunks)
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")

    queries = [
        "Who wrote Fundamentals of Wavelets?",
        "Which books are published by Wiley?",
        "Show me fiction books.",
        "Which books belong to the history category?"
    ]

    for query in queries:
        print(f"\n{'=' * 65}")
        print(f"QUERY: {query}")
        print("=" * 65)

        # Vector (semantic) search
        vec_results = collection.query(query_texts=[query], n_results=6)
        vec_hits = [
            {"id": vid, "document": vdoc}
            for vid, vdoc in zip(
                vec_results["ids"][0],
                vec_results["documents"][0],
            )
        ]
        print(f"\n  🔍 Vector search:   {len(vec_hits)} candidates")

        # BM25 keyword search
        bm25_hits = bm25_index.search(query, top_k=6)
        print(f"  🔑 BM25 search:     {len(bm25_hits)} candidates")

        # Hybrid RRF fusion
        hybrid = reciprocal_rank_fusion(vec_hits, bm25_hits, top_k=6)
        print(f"  ⚡ RRF fusion:      {len(hybrid)} merged candidates")

        # Cross-encoder rerank
        final_docs = rerank(query, hybrid, top_k=3)

        # Build prompt
        prompt = build_rag_prompt(query, final_docs)

        print("\n  📝 Top-3 chunks after reranking:")
        for i, doc in enumerate(final_docs, 1):
            print(f"    [{i}] {doc['text'][:110]}...")

        print("\n  📋 Prompt sent to LLM (first 300 chars):")
        print(f"    {prompt[:300]}...")
        print("\n  [In production: call your LLM API here with the full prompt]")

        # Evaluate
        metrics = evaluate(query, final_docs, embed_model)
        print(f"\n  📊 Retrieval metrics: {metrics}")
        if metrics["context_precision"] < 0.5:
            print("  ⚠️  Low precision — consider increasing ef or lowering chunk overlap")
        else:
            print("  ✅  Retrieval quality looks good")

    print("\n" + "=" * 65)
    print("Lab complete. Key takeaways:")
    print("  1. Vector search finds semantic matches — misses exact terms")
    print("  2. BM25 finds exact keyword matches — misses paraphrases")
    print("  3. RRF fuses both result lists without needing to tune weights")
    print("  4. Cross-encoder adds a high-precision second stage (at O(n) cost)")
    print("  5. Always measure: context precision, recall, faithfulness, relevance")
    print("=" * 65)

## ▶ Run

In [37]:
main()

  Day 3 Lab — Vector Databases and Advanced RAG Engineering

📄 211 documents → 1059 chunks after splitting

📦 Building ChromaDB vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...


  📝 Top-3 chunks after reranking:
    [1] Publisher: Wiley...
    [2] Publisher: Wiley...
    [3] Publisher: HarperCollins...

  📋 Prompt sent to LLM (first 300 chars):
    You are a data engineering expert. Answer the question strictly based
on the provided context. Do not add information not in the context.

CONTEXT:
[Source 1]: Publisher: Wiley

[Source 2]: Publisher: Wiley

[Source 3]: Publisher: HarperCollins

QUESTION: Which books are published by Wiley?

ANSWER:...

  [In production: call your LLM API here with the full prompt]

  📊 Retrieval metrics: {'context_precision': 1.0, 'avg_similarity': 0.701, 'chunks_in_context': 3}
  ✅  Retrieval quality looks good

QUERY: Show me fiction books.

  🔍 Vector search:   6 candidates
  🔑 BM25 search:     6 candidates
  ⚡ RRF fusion:      6 merged candidates
  🎯 Cross-encoder reranking 6 candidates...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


  📝 Top-3 chunks after reranking:
    [1] Author: Rowling, J K. Genre: fiction....
    [2] S.. Genre: fiction....
    [3] G.. Genre: fiction....

  📋 Prompt sent to LLM (first 300 chars):
    You are a data engineering expert. Answer the question strictly based
on the provided context. Do not add information not in the context.

CONTEXT:
[Source 1]: Author: Rowling, J K. Genre: fiction.

[Source 2]: S.. Genre: fiction.

[Source 3]: G.. Genre: fiction.

QUESTION: Show me fiction books.

A...

  [In production: call your LLM API here with the full prompt]

  📊 Retrieval metrics: {'context_precision': 1.0, 'avg_similarity': 0.598, 'chunks_in_context': 3}
  ✅  Retrieval quality looks good

QUERY: Which books belong to the history category?

  🔍 Vector search:   6 candidates
  🔑 BM25 search:     6 candidates
  ⚡ RRF fusion:      6 merged candidates
  🎯 Cross-encoder reranking 6 candidates...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]


  📝 Top-3 chunks after reranking:
    [1] Author: Fisk, Robert. Genre: history....
    [2] Author: Fisk, Robert. Genre: history....
    [3] Author: Friedman, Thomas. Genre: history....

  📋 Prompt sent to LLM (first 300 chars):
    You are a data engineering expert. Answer the question strictly based
on the provided context. Do not add information not in the context.

CONTEXT:
[Source 1]: Author: Fisk, Robert. Genre: history.

[Source 2]: Author: Fisk, Robert. Genre: history.

[Source 3]: Author: Friedman, Thomas. Genre: histo...

  [In production: call your LLM API here with the full prompt]

  📊 Retrieval metrics: {'context_precision': 1.0, 'avg_similarity': 0.533, 'chunks_in_context': 3}
  ✅  Retrieval quality looks good

Lab complete. Key takeaways:
  1. Vector search finds semantic matches — misses exact terms
  2. BM25 finds exact keyword matches — misses paraphrases
  3. RRF fuses both result lists without needing to tune weights
  4. Cross-encoder adds a high-precision second 